In [6]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Figure 1.3 (Rev 5 - Minimal Dataset Compatible)
-----------------------------------------------
修复: 适配 Minimal 数据集结构
1. 使用 'region_labels' 替代缺失的 'one_hot_loc_alex_label'。
2. 移除 argmax 解码步骤，因为 region_labels 已经是整数映射。

输入文件:
- .mat 数据集 (Minimal version)
- Freesurfer_LUT...xlsx (Label 字典)
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
import pandas as pd
import h5py
import os

# ================= 配置区域 =================

# 1. 文件路径
DATA_PATH = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat"
LUT_CSV_PATH = "/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx"

# 2. 切片选择
SLICE_IDX = 222

# 3. 模拟低分辨网格比例 (3x3 MPRAGE voxels = 1 CEST voxel)
GRID_RATIO = 3

# 4. 搜索配置
MANUAL_GRID_CENTER = None 

# 5. 排除的 Label ID
IGNORE_LABELS = [0, 28, 60] 

# 6. 特征索引
IDX_MPRAGE = 341
N_FEATURES = 341

OUTPUT_FILENAME = "Figure_1_3_v5_MinimalFixed.png"

# ================= 数据加载 =================

def load_lut(file_path):
    """加载 Label ID -> Name 映射"""
    print(f"Loading LUT: {file_path} ...")
    try:
        if file_path.endswith((".xlsx", ".xls")):
            df = pd.read_excel(file_path)
        else:
            df = pd.read_csv(file_path, sep=None, engine="python")

        # 智能查找列名
        def find_col(candidates):
            for c in candidates:
                if c in df.columns: return c
            return None

        col_id = find_col(["one_hot_loc_alex_label", "freesurfer_label", "idx"])
        col_name = find_col(["freesurfer_tissue_name", "tissue_name", "col label"])

        if not col_id or not col_name:
            print(f"[Warn] LUT columns not found. Available: {list(df.columns)}")
            return {}

        # 构建字典
        valid = pd.to_numeric(df[col_id], errors='coerce').notna()
        ids = df.loc[valid, col_id].astype(int)
        names = df.loc[valid, col_name].astype(str)
        
        lut = dict(zip(ids, names))
        print(f"Loaded {len(lut)} labels.")
        return lut
    except Exception as e:
        print(f"[Warn] Failed to load LUT: {e}")
        return {}

def load_data_and_labels(mat_path):
    """
    加载 Minimal 数据集
    根据 README:
    - data: (351, 384, 336, 256) -> 需要转置为 (384, 336, 256, 351)
    - region_labels: (384, 336, 256) -> 不需要转置，直接使用
    """
    print(f"Loading Data: {mat_path} ...")
    with h5py.File(mat_path, 'r') as f:
        # 1. 加载特征 (处理 Fortran Order 转置)
        data = f['data'][:]
        if data.shape[0] in [341, 351]: 
            # 如果特征维在第一位 (HDF5默认)，移到最后
            data = np.moveaxis(data, 0, -1)
            
        # 2. 加载标签 (Minimal 数据集使用的是 region_labels)
        if 'region_labels' in f:
            print("Found 'region_labels' (Minimal Format).")
            label_map = f['region_labels'][:]
        elif 'one_hot_loc_alex_label' in f:
            print("Found 'one_hot_loc_alex_label' (Legacy Format). Converting...")
            one_hot = f['one_hot_loc_alex_label'][:]
            # 兼容旧代码逻辑：如果是4D one-hot，转argmax
            if one_hot.ndim == 4:
                # 判断 class 维度在哪里
                axis = 0 if one_hot.shape[0] == 102 else -1
                label_map = np.argmax(one_hot, axis=axis)
            else:
                label_map = one_hot.astype(int)
        else:
            raise ValueError("Dataset missing 'region_labels' or 'one_hot_loc_alex_label'!")
            
    # 最终形状检查
    print(f"  Data: {data.shape}")
    print(f"  Labels: {label_map.shape}")
    
    return data, label_map.astype(int)

def normalize_sig(vec):
    """Z-score 归一化"""
    std = np.std(vec)
    if std < 1e-9: std = 1.0
    return (vec - np.mean(vec)) / std

# ================= 核心算法：寻找最佳混合块 =================

def find_best_mixed_block(label_slice, grid_ratio, lut):
    """寻找包含两种不同组织的 3x3 Block"""
    h, w = label_slice.shape
    best_score = -1
    best_pos = (h//2, w//2)
    best_labels = (0, 0)
    
    # 遍历 Grid
    for r in range(0, h - grid_ratio, grid_ratio):
        for c in range(0, w - grid_ratio, grid_ratio):
            block = label_slice[r:r+grid_ratio, c:c+grid_ratio]
            uniques = np.unique(block)
            valid = [u for u in uniques if u not in IGNORE_LABELS]
            
            if len(valid) >= 2:
                u1, u2 = valid[0], valid[1]
                # 计算平衡度 (min count)
                score = min(np.sum(block==u1), np.sum(block==u2))
                
                # 优先选择有意义的结构名 (可选)
                n1 = lut.get(u1, "").lower()
                n2 = lut.get(u2, "").lower()
                if "wm" in n1 or "white" in n1: score += 1
                if "ctx" in n2 or "cortex" in n2: score += 1
                
                if score > best_score:
                    best_score = score
                    best_pos = (r, c)
                    best_labels = (u1, u2)
    
    print(f"Best Mixed Block: {best_pos}, Labels: {best_labels}")
    return best_pos, best_labels

# ================= 绘图 =================

def plot_rev5(data, label_map, lut, slice_idx, output_file):
    
    # 准备切片
    img_slice = data[slice_idx, :, :, IDX_MPRAGE]
    lbl_slice = label_map[slice_idx, :, :]
    
    # 归一化显示图像
    p1, p99 = np.percentile(img_slice, [1, 99])
    img_disp = np.clip((img_slice - p1) / (p99 - p1 + 1e-8), 0, 1)

    # 确定位置
    if MANUAL_GRID_CENTER:
        r_grid = (MANUAL_GRID_CENTER[0] // GRID_RATIO) * GRID_RATIO
        c_grid = (MANUAL_GRID_CENTER[1] // GRID_RATIO) * GRID_RATIO
        block_tl = (r_grid, c_grid)
        # 获取标签
        patch = lbl_slice[r_grid:r_grid+GRID_RATIO, c_grid:c_grid+GRID_RATIO]
        valid = [u for u in np.unique(patch) if u not in IGNORE_LABELS]
        target_labels = tuple(valid[:2]) if len(valid)>=2 else (valid[0], valid[0])
    else:
        block_tl, target_labels = find_best_mixed_block(lbl_slice, GRID_RATIO, lut)

    # 定义 Crop 区域
    margin = 25
    r_start = max(0, block_tl[0] - margin)
    r_end = min(img_slice.shape[0], block_tl[0] + GRID_RATIO + margin)
    c_start = max(0, block_tl[1] - margin)
    c_end = min(img_slice.shape[1], block_tl[1] + GRID_RATIO + margin)
    
    img_crop = img_disp[r_start:r_end, c_start:c_end]
    lbl_crop = lbl_slice[r_start:r_end, c_start:c_end]

    # 提取光谱
    r_b, c_b = block_tl
    coords_A, coords_B = [], []
    
    for i in range(GRID_RATIO):
        for j in range(GRID_RATIO):
            rr, cc = r_b + i, c_b + j
            val = lbl_slice[rr, cc]
            if val == target_labels[0]: coords_A.append((rr, cc))
            elif val == target_labels[1]: coords_B.append((rr, cc))
            
    vecs_A = [data[slice_idx, r, c, :N_FEATURES] for r, c in coords_A]
    vecs_B = [data[slice_idx, r, c, :N_FEATURES] for r, c in coords_B]
    # 混合信号 = 整个 Block 的均值
    vec_mixed = np.mean(data[slice_idx, r_b:r_b+GRID_RATIO, c_b:c_b+GRID_RATIO, :N_FEATURES], axis=(0,1))
    
    sig_A = normalize_sig(np.mean(vecs_A, axis=0)) if vecs_A else np.zeros(N_FEATURES)
    sig_B = normalize_sig(np.mean(vecs_B, axis=0)) if vecs_B else np.zeros(N_FEATURES)
    sig_M = normalize_sig(vec_mixed)
    
    name_A = lut.get(target_labels[0], f"Tissue A ({target_labels[0]})")
    name_B = lut.get(target_labels[1], f"Tissue B ({target_labels[1]})")

    # 绘图
    fig = plt.figure(figsize=(18, 6), facecolor='white')
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1.2, 1.5], wspace=0.25)
    
    # A
    ax1 = fig.add_subplot(gs[0])
    ax1.imshow(img_disp, cmap='gray', origin='upper')
    rect = patches.Rectangle((c_start, r_start), c_end-c_start, r_end-r_start, lw=1.5, edgecolor='lime', facecolor='none')
    ax1.add_patch(rect)
    ax1.add_patch(patches.Rectangle((c_b, r_b), GRID_RATIO, GRID_RATIO, lw=2, edgecolor='red', facecolor='none'))
    ax1.set_title("(A) Full Slice", fontweight='bold')
    ax1.axis('off')
    
    # B
    ax2 = fig.add_subplot(gs[1])
    ax2.imshow(img_crop, cmap='gray', origin='upper', extent=[c_start, c_end, r_end, r_start])
    
    # Color Overlay
    overlay = np.zeros((lbl_crop.shape[0], lbl_crop.shape[1], 4))
    overlay[lbl_crop == target_labels[0]] = [0, 0, 1, 0.4] # Blue
    overlay[lbl_crop == target_labels[1]] = [0, 1, 0, 0.4] # Green
    ax2.imshow(overlay, origin='upper', extent=[c_start, c_end, r_end, r_start])
    
    # Grid Lines
    grid_sc = (c_start // GRID_RATIO) * GRID_RATIO
    grid_sr = (r_start // GRID_RATIO) * GRID_RATIO
    for c in range(grid_sc, c_end+1, GRID_RATIO):
        ax2.axvline(c - 0.5, color='cyan', lw=0.5, alpha=0.5)
    for r in range(grid_sr, r_end+1, GRID_RATIO):
        ax2.axhline(r - 0.5, color='cyan', lw=0.5, alpha=0.5)
        
    ax2.add_patch(patches.Rectangle((c_b-0.5, r_b-0.5), GRID_RATIO, GRID_RATIO, lw=2.5, edgecolor='red', facecolor='none'))
    ax2.set_title("(B) Mixed Voxel (Red Box)", fontweight='bold')
    ax2.axis('off')
    
    # Legend B
    from matplotlib.lines import Line2D
    legs = [Line2D([0],[0], color='blue', lw=4, alpha=0.5), Line2D([0],[0], color='green', lw=4, alpha=0.5)]
    ax2.legend(legs, [name_A, name_B], loc='lower center', bbox_to_anchor=(0.5, -0.15), fontsize=8)

    # C
    ax3 = fig.add_subplot(gs[2])
    x = np.arange(N_FEATURES)
    ax3.plot(x, sig_A, 'b-', alpha=0.5, lw=1, label=f'Pure {name_A}')
    ax3.plot(x, sig_B, 'g-', alpha=0.5, lw=1, label=f'Pure {name_B}')
    ax3.plot(x, sig_M, 'r-', alpha=1.0, lw=2, label='Mixed Signal')
    
    ax3.set_title("(C) Spectral Mixing", fontweight='bold')
    ax3.set_ylabel("Normalized Intensity")
    ax3.legend(loc='upper right', fontsize=8)
    
    # Annotations
    trans = ax3.get_xaxis_transform()
    ax3.text(7, -0.1, "QTI", transform=trans, ha='center', size=8)
    ax3.text(120, -0.1, "Diffusion", transform=trans, ha='center', size=8)
    ax3.text(280, -0.1, "CEST", transform=trans, ha='center', size=8)
    for l in [15, 225]: ax3.axvline(l, color='gray', ls='--', alpha=0.3)
    ax3.set_ylim(-3, 4)
    
    plt.tight_layout()
    plt.savefig(output_file, dpi=300)
    print(f"Generated: {output_file}")
    plt.close()

if __name__ == "__main__":
    lut = load_lut(LUT_CSV_PATH)
    data, labels = load_data_and_labels(DATA_PATH)
    plot_rev5(data, labels, lut, SLICE_IDX, OUTPUT_FILENAME)

Loading LUT: /home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx ...
Loaded 102 labels.
Loading Data: /home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat ...
Found 'region_labels' (Minimal Format).
  Data: (384, 336, 256, 351)
  Labels: (384, 336, 256)
Best Mixed Block: (183, 135), Labels: (np.int64(14), np.int64(40))


/tmp/ipykernel_28679/1812641952.py:279: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Generated: Figure_1_3_v5_MinimalFixed.png


In [9]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Figure 1.3 (Rev 6 - Minimal + Non-normalized Spectra)
-----------------------------------------------------
Changes vs Rev5:
1) Minimal dataset compatible:
   - Prefer 'region_labels' as integer label map
   - Fallback to legacy 'one_hot_loc_alex_label' (argmax only if 4D)
2) Robust mixed-block selection:
   - In each GRID_RATIO x GRID_RATIO block, choose top-2 labels by frequency (excluding IGNORE_LABELS)
   - Fix boundary scanning (include last blocks)
3) Manual grid center safe fallback:
   - If manual patch contains only ignored labels, fallback to auto-search
4) Panel (C) spectra: NO z-score standardization.
   - Default: robust display scaling (percentile-based) to enhance visibility.
   - Option: plot raw intensities without any scaling.

Inputs:
- Minimal .mat dataset (HDF5)
- Freesurfer LUT .xlsx/.csv for label name mapping

Output:
- Figure_1_3_v6_Minimal_NonNorm.png
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec
import pandas as pd
import h5py
import os

# ================= Configuration =================

DATA_PATH = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat"
LUT_CSV_PATH = "/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx"

SLICE_IDX = 222

# Conceptual mapping: GRID_RATIO x GRID_RATIO MPRAGE voxels ~= 1 low-res voxel
GRID_RATIO = 3

# If you want to force a specific location (row, col) in full-res slice, set it here, e.g. (140, 120).
# Script will snap to the top-left of the corresponding GRID_RATIO block.
MANUAL_GRID_CENTER = None  # e.g. (140, 120)

# Labels to ignore (background, unknown, etc.)
IGNORE_LABELS = [0, 28, 60]

# Feature indexing
IDX_MPRAGE = 341          # channel index used for panel (A)(B) background anatomy
N_FEATURES = 341          # number of features shown in spectrum (C)

OUTPUT_FILENAME = "Figure_1_3_v6_Minimal_NonNorm.png"

# Panel C options:
PLOT_SPECTRA_RAW = False
# If False: use robust scaling to [0,1] per-curve using percentiles (recommended for visibility)
SPECTRA_PCTL_LOW = 1
SPECTRA_PCTL_HIGH = 99

# Crop margin around selected block for panel (B)
CROP_MARGIN = 25

# ================= Utilities =================

def load_lut(file_path):
    """Load label ID -> name mapping from .xlsx or .csv, with flexible column detection."""
    print(f"Loading LUT: {file_path} ...")
    try:
        if file_path.endswith((".xlsx", ".xls")):
            df = pd.read_excel(file_path)
        else:
            df = pd.read_csv(file_path, sep=None, engine="python")

        def find_col(candidates):
            for c in candidates:
                if c in df.columns:
                    return c
            return None

        # Your LUT has: tissue_name, one_hot_loc_alex_label, nii_filename, freesurfer_label, freesurfer_tissue_name, R,G,B
        col_id = find_col(["one_hot_loc_alex_label", "freesurfer_label", "idx"])
        col_name = find_col(["freesurfer_tissue_name", "tissue_name", "col label"])

        if (col_id is None) or (col_name is None):
            print(f"[Warn] LUT columns not found. Available columns: {list(df.columns)}")
            return {}

        valid = pd.to_numeric(df[col_id], errors="coerce").notna()
        ids = df.loc[valid, col_id].astype(int).values
        names = df.loc[valid, col_name].astype(str).values

        lut = dict(zip(ids, names))
        print(f"Loaded {len(lut)} labels from LUT.")
        return lut

    except Exception as e:
        print(f"[Warn] Failed to load LUT: {e}")
        return {}


def load_data_and_labels(mat_path):
    """
    Load Minimal dataset.
    Expected (per your README):
      - data: (351, 384, 336, 256)  -> moveaxis to (384, 336, 256, 351)
      - region_labels: (384, 336, 256)  -> use directly
    Also supports legacy one_hot_loc_alex_label if present.
    """
    print(f"Loading Data: {mat_path} ...")
    with h5py.File(mat_path, "r") as f:
        if "data" not in f:
            raise ValueError("Dataset missing 'data'!")

        data = f["data"][:]
        # Move feature dim to last if it is first (common in HDF5 export)
        if data.shape[0] in [341, 351]:
            data = np.moveaxis(data, 0, -1)

        if "region_labels" in f:
            print("Found 'region_labels' (Minimal Format).")
            label_map = f["region_labels"][:]
        elif "one_hot_loc_alex_label" in f:
            print("Found 'one_hot_loc_alex_label' (Legacy Format). Converting if needed...")
            one_hot = f["one_hot_loc_alex_label"][:]
            if one_hot.ndim == 4:
                # Determine class axis: if first dim is 102, class axis=0 else last
                axis = 0 if one_hot.shape[0] == 102 else -1
                label_map = np.argmax(one_hot, axis=axis)
            else:
                label_map = one_hot.astype(int)
        else:
            raise ValueError("Dataset missing 'region_labels' or 'one_hot_loc_alex_label'!")

    print(f"  Data shape:   {data.shape}")
    print(f"  Labels shape: {label_map.shape}")

    return data, label_map.astype(int)


def robust_scale_01(vec, p_low=1, p_high=99):
    """
    Robustly scale a 1D vector to [0,1] using percentiles.
    This is NOT z-score and keeps relative amplitude patterns visible.
    """
    v = np.asarray(vec, dtype=np.float64)
    lo, hi = np.percentile(v, [p_low, p_high])
    den = hi - lo
    if den < 1e-12:
        den = 1.0
    out = (v - lo) / den
    return np.clip(out, 0.0, 1.0)


def pick_top2_labels_by_frequency(block, ignore_labels):
    """Return (u1,u2,score) where u1/u2 are top-2 labels by count excluding ignore_labels."""
    vals, counts = np.unique(block, return_counts=True)
    mask = ~np.isin(vals, ignore_labels)
    vals, counts = vals[mask], counts[mask]
    if vals.size < 2:
        return None
    order = np.argsort(counts)[::-1]
    u1, u2 = int(vals[order[0]]), int(vals[order[1]])
    score = int(min(counts[order[0]], counts[order[1]]))
    return u1, u2, score


def find_best_mixed_block(label_slice, grid_ratio, lut, ignore_labels):
    """
    Scan blocks aligned with grid_ratio.
    Pick the block that contains >=2 valid labels, selecting the top-2 by frequency.
    Score is min(countA, countB) + small optional preference based on LUT names.
    """
    h, w = label_slice.shape
    best_score = -1
    best_pos = (h // 2, w // 2)
    best_labels = (0, 0)

    for r in range(0, h - grid_ratio + 1, grid_ratio):
        for c in range(0, w - grid_ratio + 1, grid_ratio):
            block = label_slice[r:r+grid_ratio, c:c+grid_ratio]
            top2 = pick_top2_labels_by_frequency(block, ignore_labels)
            if top2 is None:
                continue

            u1, u2, score = top2

            # Optional tiny bias to more meaningful-looking pairs (doesn't dominate)
            n1 = lut.get(u1, "").lower()
            n2 = lut.get(u2, "").lower()
            if ("wm" in n1) or ("white" in n1):
                score += 1
            if ("ctx" in n2) or ("cortex" in n2):
                score += 1

            if score > best_score:
                best_score = score
                best_pos = (r, c)
                best_labels = (u1, u2)

    print(f"Best Mixed Block: {best_pos}, Labels: {best_labels}, Score: {best_score}")
    return best_pos, best_labels


# ================= Plotting =================

def plot_rev6(data, label_map, lut, slice_idx, output_file):
    # Basic checks
    if data.ndim != 4:
        raise ValueError(f"Expected data 4D (X,Y,Z,C). Got {data.ndim}D.")
    if label_map.ndim != 3:
        raise ValueError(f"Expected labels 3D (X,Y,Z). Got {label_map.ndim}D.")
    if not (0 <= slice_idx < data.shape[0]):
        raise ValueError(f"SLICE_IDX={slice_idx} out of range [0, {data.shape[0]-1}]")
    if IDX_MPRAGE < 0 or IDX_MPRAGE >= data.shape[-1]:
        raise ValueError(f"IDX_MPRAGE={IDX_MPRAGE} out of range [0, {data.shape[-1]-1}]")
    if N_FEATURES > data.shape[-1]:
        raise ValueError(f"N_FEATURES={N_FEATURES} > available channels {data.shape[-1]}")

    # Prepare slice
    img_slice = data[slice_idx, :, :, IDX_MPRAGE]
    lbl_slice = label_map[slice_idx, :, :]

    # Display normalization for anatomical background (A)(B) ONLY
    p1, p99 = np.percentile(img_slice, [1, 99])
    den = (p99 - p1)
    if den < 1e-8:
        den = 1.0
    img_disp = np.clip((img_slice - p1) / den, 0, 1)

    # Determine block
    if MANUAL_GRID_CENTER is not None:
        rr0 = (MANUAL_GRID_CENTER[0] // GRID_RATIO) * GRID_RATIO
        cc0 = (MANUAL_GRID_CENTER[1] // GRID_RATIO) * GRID_RATIO
        block_tl = (int(rr0), int(cc0))

        patch = lbl_slice[block_tl[0]:block_tl[0]+GRID_RATIO, block_tl[1]:block_tl[1]+GRID_RATIO]
        valid = [int(u) for u in np.unique(patch) if int(u) not in IGNORE_LABELS]

        if len(valid) >= 2:
            target_labels = (valid[0], valid[1])
        elif len(valid) == 1:
            target_labels = (valid[0], valid[0])
        else:
            print("[Warn] Manual patch contains only ignored labels; fallback to auto-search.")
            block_tl, target_labels = find_best_mixed_block(lbl_slice, GRID_RATIO, lut, IGNORE_LABELS)
    else:
        block_tl, target_labels = find_best_mixed_block(lbl_slice, GRID_RATIO, lut, IGNORE_LABELS)

    r_b, c_b = block_tl

    # Crop region for panel (B)
    r_start = max(0, r_b - CROP_MARGIN)
    r_end = min(img_slice.shape[0], r_b + GRID_RATIO + CROP_MARGIN)
    c_start = max(0, c_b - CROP_MARGIN)
    c_end = min(img_slice.shape[1], c_b + GRID_RATIO + CROP_MARGIN)

    img_crop = img_disp[r_start:r_end, c_start:c_end]
    lbl_crop = lbl_slice[r_start:r_end, c_start:c_end]

    # Collect coordinates for tissue A/B inside the block
    coords_A, coords_B = [], []
    for i in range(GRID_RATIO):
        for j in range(GRID_RATIO):
            rr, cc = r_b + i, c_b + j
            val = int(lbl_slice[rr, cc])
            if val == int(target_labels[0]):
                coords_A.append((rr, cc))
            elif val == int(target_labels[1]):
                coords_B.append((rr, cc))

    # Extract spectra (NO z-score)
    vecs_A = [data[slice_idx, r, c, :N_FEATURES] for (r, c) in coords_A]
    vecs_B = [data[slice_idx, r, c, :N_FEATURES] for (r, c) in coords_B]
    vec_mixed = np.mean(data[slice_idx, r_b:r_b+GRID_RATIO, c_b:c_b+GRID_RATIO, :N_FEATURES], axis=(0, 1))

    sig_A_raw = np.mean(vecs_A, axis=0) if len(vecs_A) else np.zeros(N_FEATURES, dtype=np.float64)
    sig_B_raw = np.mean(vecs_B, axis=0) if len(vecs_B) else np.zeros(N_FEATURES, dtype=np.float64)
    sig_M_raw = vec_mixed.astype(np.float64)

    # For visualization in panel (C):
    if PLOT_SPECTRA_RAW:
        sig_A, sig_B, sig_M = sig_A_raw, sig_B_raw, sig_M_raw
        y_label = "Intensity (raw units)"
    else:
        sig_A = robust_scale_01(sig_A_raw, SPECTRA_PCTL_LOW, SPECTRA_PCTL_HIGH)
        sig_B = robust_scale_01(sig_B_raw, SPECTRA_PCTL_LOW, SPECTRA_PCTL_HIGH)
        sig_M = robust_scale_01(sig_M_raw, SPECTRA_PCTL_LOW, SPECTRA_PCTL_HIGH)
        y_label = f"Intensity (robust scaled to [0,1], p{SPECTRA_PCTL_LOW}–p{SPECTRA_PCTL_HIGH})"

    name_A = lut.get(int(target_labels[0]), f"Tissue A ({int(target_labels[0])})")
    name_B = lut.get(int(target_labels[1]), f"Tissue B ({int(target_labels[1])})")

    # ===== Plot layout =====
    fig = plt.figure(figsize=(18, 6), facecolor="white")
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1.2, 1.5], wspace=0.25)

    # (A) Full slice
    ax1 = fig.add_subplot(gs[0])
    ax1.imshow(img_disp, cmap="gray", origin="upper")
    ax1.add_patch(
        patches.Rectangle(
            (c_start, r_start),
            c_end - c_start,
            r_end - r_start,
            lw=1.5,
            edgecolor="lime",
            facecolor="none",
        )
    )
    ax1.add_patch(
        patches.Rectangle(
            (c_b, r_b),
            GRID_RATIO,
            GRID_RATIO,
            lw=2,
            edgecolor="red",
            facecolor="none",
        )
    )
    ax1.set_title("(A) Full Slice", fontweight="bold")
    ax1.axis("off")

    # (B) Cropped + label overlay + grid lines
    ax2 = fig.add_subplot(gs[1])
    ax2.imshow(img_crop, cmap="gray", origin="upper", extent=[c_start, c_end, r_end, r_start])

    overlay = np.zeros((lbl_crop.shape[0], lbl_crop.shape[1], 4), dtype=np.float32)
    overlay[lbl_crop == int(target_labels[0])] = [0, 0, 1, 0.40]  # blue
    overlay[lbl_crop == int(target_labels[1])] = [0, 1, 0, 0.40]  # green
    ax2.imshow(overlay, origin="upper", extent=[c_start, c_end, r_end, r_start])

    # grid lines (aligned)
    grid_sc = (c_start // GRID_RATIO) * GRID_RATIO
    grid_sr = (r_start // GRID_RATIO) * GRID_RATIO
    for c in range(grid_sc, c_end + 1, GRID_RATIO):
        ax2.axvline(c - 0.5, lw=0.5, alpha=0.5, color="cyan")
    for r in range(grid_sr, r_end + 1, GRID_RATIO):
        ax2.axhline(r - 0.5, lw=0.5, alpha=0.5, color="cyan")

    ax2.add_patch(
        patches.Rectangle(
            (c_b - 0.5, r_b - 0.5),
            GRID_RATIO,
            GRID_RATIO,
            lw=2.5,
            edgecolor="red",
            facecolor="none",
        )
    )
    ax2.set_title("(B) Mixed Block (Red Box)", fontweight="bold")
    ax2.axis("off")

    # legend
    from matplotlib.lines import Line2D
    legs = [
        Line2D([0], [0], color="blue", lw=4, alpha=0.5),
        Line2D([0], [0], color="green", lw=4, alpha=0.5),
    ]
    ax2.legend(legs, [name_A, name_B], loc="lower center", bbox_to_anchor=(0.5, -0.15), fontsize=8)

    # (C) Spectral curves (non-normalized)
    ax3 = fig.add_subplot(gs[2])
    x = np.arange(N_FEATURES, dtype=int)
    ax3.plot(x, sig_A, "-", alpha=0.6, lw=1, label=f"Pure {name_A}")
    ax3.plot(x, sig_B, "-", alpha=0.6, lw=1, label=f"Pure {name_B}")
    ax3.plot(x, sig_M, "-", alpha=1.0, lw=2, label="Mixed (block mean)")

    ax3.set_title("(C) Spectral Mixing (No z-score)", fontweight="bold")
    ax3.set_ylabel(y_label)
    ax3.set_xlabel("Feature Index")
    ax3.legend(loc="upper right", fontsize=8)

    # modality boundary hints (same as your previous)
    trans = ax3.get_xaxis_transform()
    ax3.text(7, -0.12, "QTI", transform=trans, ha="center", size=8)
    ax3.text(120, -0.12, "Diffusion", transform=trans, ha="center", size=8)
    ax3.text(280, -0.12, "CEST", transform=trans, ha="center", size=8)
    for l in [15, 225]:
        ax3.axvline(l, color="gray", ls="--", alpha=0.3)

    # y-limits
    if PLOT_SPECTRA_RAW:
        # let matplotlib auto-scale but keep a tiny margin
        y_min = np.nanmin([sig_A.min(), sig_B.min(), sig_M.min()])
        y_max = np.nanmax([sig_A.max(), sig_B.max(), sig_M.max()])
        if np.isfinite(y_min) and np.isfinite(y_max) and (y_max - y_min) > 1e-12:
            pad = 0.05 * (y_max - y_min)
            ax3.set_ylim(y_min - pad, y_max + pad)
    else:
        ax3.set_ylim(-0.05, 1.05)

    plt.tight_layout()
    plt.savefig(output_file, dpi=300)
    print(f"Generated: {output_file}")
    plt.close()


if __name__ == "__main__":
    lut = load_lut(LUT_CSV_PATH)
    data, labels = load_data_and_labels(DATA_PATH)
    plot_rev6(data, labels, lut, SLICE_IDX, OUTPUT_FILENAME)


Loading LUT: /home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx ...
Loaded 102 labels from LUT.
Loading Data: /home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat ...
Found 'region_labels' (Minimal Format).
  Data shape:   (384, 336, 256, 351)
  Labels shape: (384, 336, 256)
Best Mixed Block: (36, 111), Labels: (89, 55), Score: 6


/tmp/ipykernel_28679/3391445288.py:397: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Generated: Figure_1_3_v6_Minimal_NonNorm.png


In [11]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Figure 1.3 (Rev 7 - Minimal + Fixed Block + Non-zscore + Pretty Names)
----------------------------------------------------------------------
Changes vs Rev6:
1) FIXED block support (freeze slice + block top-left), so the same region/voxels are always used.
2) Robust mixed-block logic kept (top-2 labels by frequency) for both auto and fixed cases.
3) Legend display names are formatted (remove -rh-/-lh-, optional remove wm/ctx prefix) WITHOUT changing labels.

Inputs:
- Minimal .mat dataset (HDF5)
- Freesurfer LUT .xlsx/.csv for label name mapping

Output:
- Figure_1_3_v7_FixedBlock_PrettyNames.png
"""

import os
import re
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.gridspec as gridspec

# ================= Configuration =================

DATA_PATH = "/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat"
LUT_CSV_PATH = "/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx"

# --- Freeze region (highest priority) ---
USE_FIXED_BLOCK = True
FIXED_SLICE_IDX = 222
FIXED_BLOCK_TL = (36, 111)  # (row, col) top-left of GRID_RATIO block

# If you prefer manual picking (row, col) in full-res slice, set it here and set USE_FIXED_BLOCK=False
MANUAL_GRID_CENTER = None  # e.g. (140, 120)

# Conceptual mapping: GRID_RATIO x GRID_RATIO MPRAGE voxels ~= 1 low-res voxel
GRID_RATIO = 3

# Labels to ignore (background, unknown, etc.)
IGNORE_LABELS = [0, 28, 60]

# Feature indexing
IDX_MPRAGE = 341          # channel index used for panel (A)(B) background anatomy
N_FEATURES = 341          # number of features shown in spectrum (C)

# Output
OUTPUT_FILENAME = "Figure_1_3_v7_FixedBlock_PrettyNames.png"

# Panel C options: NO z-score standardization
PLOT_SPECTRA_RAW = False          # True = raw intensities; False = robust scaled to [0,1] for visibility
SPECTRA_PCTL_LOW = 1
SPECTRA_PCTL_HIGH = 99

# Crop margin around selected block for panel (B)
CROP_MARGIN = 25

# Pretty display name options (ONLY for legend text)
STRIP_HEMISPHERE = True           # remove -rh- / -lh-
STRIP_PREFIX_WM_CTX = False       # remove leading "wm-" / "ctx-" from legend names

# ================= Utilities =================

def load_lut(file_path: str) -> dict:
    """Load label ID -> name mapping from .xlsx or .csv, with flexible column detection."""
    print(f"Loading LUT: {file_path} ...")
    try:
        if file_path.endswith((".xlsx", ".xls")):
            df = pd.read_excel(file_path)
        else:
            df = pd.read_csv(file_path, sep=None, engine="python")

        def find_col(candidates):
            for c in candidates:
                if c in df.columns:
                    return c
            return None

        col_id = find_col(["one_hot_loc_alex_label", "freesurfer_label", "idx"])
        col_name = find_col(["freesurfer_tissue_name", "tissue_name", "col label"])

        if (col_id is None) or (col_name is None):
            print(f"[Warn] LUT columns not found. Available columns: {list(df.columns)}")
            return {}

        valid = pd.to_numeric(df[col_id], errors="coerce").notna()
        ids = df.loc[valid, col_id].astype(int).values
        names = df.loc[valid, col_name].astype(str).values

        lut = dict(zip(ids, names))
        print(f"Loaded {len(lut)} labels from LUT.")
        return lut

    except Exception as e:
        print(f"[Warn] Failed to load LUT: {e}")
        return {}


def load_data_and_labels(mat_path: str):
    """
    Load Minimal dataset.
    Expected:
      - data: (351, 384, 336, 256)  -> moveaxis to (384, 336, 256, 351)
      - region_labels: (384, 336, 256)  -> use directly
    Also supports legacy one_hot_loc_alex_label if present.
    """
    print(f"Loading Data: {mat_path} ...")
    with h5py.File(mat_path, "r") as f:
        if "data" not in f:
            raise ValueError("Dataset missing 'data'!")

        data = f["data"][:]
        if data.shape[0] in [341, 351]:
            data = np.moveaxis(data, 0, -1)

        if "region_labels" in f:
            print("Found 'region_labels' (Minimal Format).")
            label_map = f["region_labels"][:]
        elif "one_hot_loc_alex_label" in f:
            print("Found 'one_hot_loc_alex_label' (Legacy Format). Converting if needed...")
            one_hot = f["one_hot_loc_alex_label"][:]
            if one_hot.ndim == 4:
                axis = 0 if one_hot.shape[0] == 102 else -1
                label_map = np.argmax(one_hot, axis=axis)
            else:
                label_map = one_hot.astype(int)
        else:
            raise ValueError("Dataset missing 'region_labels' or 'one_hot_loc_alex_label'!")

    print(f"  Data shape:   {data.shape}")
    print(f"  Labels shape: {label_map.shape}")

    return data, label_map.astype(int)


def robust_scale_01(vec, p_low=1, p_high=99):
    """Robustly scale 1D vector to [0,1] using percentiles (NOT z-score)."""
    v = np.asarray(vec, dtype=np.float64)
    lo, hi = np.percentile(v, [p_low, p_high])
    den = hi - lo
    if den < 1e-12:
        den = 1.0
    out = (v - lo) / den
    return np.clip(out, 0.0, 1.0)


def pretty_label_name(name: str, strip_hemi=True, strip_prefix=False) -> str:
    """Format LUT names for figure display ONLY."""
    if name is None:
        return ""
    s = str(name)

    # Optionally strip "wm-" / "ctx-" prefix
    if strip_prefix:
        s = re.sub(r"^(wm|ctx)-", "", s)

    # Strip hemisphere tag
    if strip_hemi:
        s = s.replace("-rh-", "-").replace("-lh-", "-")

    # Keep hyphen style consistent
    s = s.replace("_", "-")
    return s


def pick_top2_labels_by_frequency(block: np.ndarray, ignore_labels) -> tuple | None:
    """Return (u1,u2,score) where u1/u2 are top-2 labels by count excluding ignore_labels."""
    vals, counts = np.unique(block, return_counts=True)
    mask = ~np.isin(vals, ignore_labels)
    vals, counts = vals[mask], counts[mask]
    if vals.size < 2:
        return None
    order = np.argsort(counts)[::-1]
    u1, u2 = int(vals[order[0]]), int(vals[order[1]])
    score = int(min(counts[order[0]], counts[order[1]]))
    return u1, u2, score


def find_best_mixed_block(label_slice: np.ndarray, grid_ratio: int, lut: dict, ignore_labels):
    """
    Scan blocks aligned with grid_ratio.
    Choose block with >=2 valid labels; select top-2 by frequency.
    Score = min(countA, countB) + tiny optional preference using LUT names.
    """
    h, w = label_slice.shape
    best_score = -1
    best_pos = (h // 2, w // 2)
    best_labels = (0, 0)

    for r in range(0, h - grid_ratio + 1, grid_ratio):
        for c in range(0, w - grid_ratio + 1, grid_ratio):
            block = label_slice[r:r + grid_ratio, c:c + grid_ratio]
            top2 = pick_top2_labels_by_frequency(block, ignore_labels)
            if top2 is None:
                continue

            u1, u2, score = top2

            n1 = lut.get(u1, "").lower()
            n2 = lut.get(u2, "").lower()
            if ("wm" in n1) or ("white" in n1):
                score += 1
            if ("ctx" in n2) or ("cortex" in n2):
                score += 1

            if score > best_score:
                best_score = score
                best_pos = (r, c)
                best_labels = (u1, u2)

    print(f"Best Mixed Block: {best_pos}, Labels: {best_labels}, Score: {best_score}")
    return best_pos, best_labels


# ================= Plotting =================

def plot_rev7(data, label_map, lut, slice_idx, output_file):
    # Basic checks
    if data.ndim != 4:
        raise ValueError(f"Expected data 4D (X,Y,Z,C). Got {data.ndim}D.")
    if label_map.ndim != 3:
        raise ValueError(f"Expected labels 3D (X,Y,Z). Got {label_map.ndim}D.")
    if IDX_MPRAGE < 0 or IDX_MPRAGE >= data.shape[-1]:
        raise ValueError(f"IDX_MPRAGE={IDX_MPRAGE} out of range [0, {data.shape[-1]-1}]")
    if N_FEATURES > data.shape[-1]:
        raise ValueError(f"N_FEATURES={N_FEATURES} > available channels {data.shape[-1]}")

    # Freeze slice if requested
    if USE_FIXED_BLOCK:
        slice_idx = FIXED_SLICE_IDX

    if not (0 <= slice_idx < data.shape[0]):
        raise ValueError(f"SLICE_IDX={slice_idx} out of range [0, {data.shape[0]-1}]")

    # Prepare slice
    img_slice = data[slice_idx, :, :, IDX_MPRAGE]
    lbl_slice = label_map[slice_idx, :, :]

    # Display normalization for anatomical background (A)(B) ONLY
    p1, p99 = np.percentile(img_slice, [1, 99])
    den = (p99 - p1)
    if den < 1e-8:
        den = 1.0
    img_disp = np.clip((img_slice - p1) / den, 0, 1)

    # Determine block (priority: fixed > manual center > auto)
    if USE_FIXED_BLOCK:
        block_tl = (int(FIXED_BLOCK_TL[0]), int(FIXED_BLOCK_TL[1]))
        r_b, c_b = block_tl

        # Safety: ensure within bounds
        if (r_b < 0) or (c_b < 0) or (r_b + GRID_RATIO > lbl_slice.shape[0]) or (c_b + GRID_RATIO > lbl_slice.shape[1]):
            raise ValueError(f"FIXED_BLOCK_TL {block_tl} out of bounds for slice shape {lbl_slice.shape} with GRID_RATIO={GRID_RATIO}")

        patch = lbl_slice[r_b:r_b + GRID_RATIO, c_b:c_b + GRID_RATIO]
        top2 = pick_top2_labels_by_frequency(patch, IGNORE_LABELS)
        if top2 is None:
            raise ValueError(f"Fixed block {block_tl} contains <2 valid labels (after IGNORE_LABELS).")
        u1, u2, _ = top2
        target_labels = (u1, u2)

    elif MANUAL_GRID_CENTER is not None:
        rr0 = (MANUAL_GRID_CENTER[0] // GRID_RATIO) * GRID_RATIO
        cc0 = (MANUAL_GRID_CENTER[1] // GRID_RATIO) * GRID_RATIO
        block_tl = (int(rr0), int(cc0))
        r_b, c_b = block_tl

        patch = lbl_slice[r_b:r_b + GRID_RATIO, c_b:c_b + GRID_RATIO]
        valid = [int(u) for u in np.unique(patch) if int(u) not in IGNORE_LABELS]
        if len(valid) >= 2:
            target_labels = (valid[0], valid[1])
        elif len(valid) == 1:
            target_labels = (valid[0], valid[0])
        else:
            print("[Warn] Manual patch contains only ignored labels; fallback to auto-search.")
            block_tl, target_labels = find_best_mixed_block(lbl_slice, GRID_RATIO, lut, IGNORE_LABELS)
            r_b, c_b = block_tl

    else:
        block_tl, target_labels = find_best_mixed_block(lbl_slice, GRID_RATIO, lut, IGNORE_LABELS)
        r_b, c_b = block_tl

    # Crop region for panel (B)
    r_start = max(0, r_b - CROP_MARGIN)
    r_end = min(img_slice.shape[0], r_b + GRID_RATIO + CROP_MARGIN)
    c_start = max(0, c_b - CROP_MARGIN)
    c_end = min(img_slice.shape[1], c_b + GRID_RATIO + CROP_MARGIN)

    img_crop = img_disp[r_start:r_end, c_start:c_end]
    lbl_crop = lbl_slice[r_start:r_end, c_start:c_end]

    # Collect coordinates for tissue A/B inside the block
    coords_A, coords_B = [], []
    for i in range(GRID_RATIO):
        for j in range(GRID_RATIO):
            rr, cc = r_b + i, c_b + j
            val = int(lbl_slice[rr, cc])
            if val == int(target_labels[0]):
                coords_A.append((rr, cc))
            elif val == int(target_labels[1]):
                coords_B.append((rr, cc))

    # Extract spectra (NO z-score)
    vecs_A = [data[slice_idx, r, c, :N_FEATURES] for (r, c) in coords_A]
    vecs_B = [data[slice_idx, r, c, :N_FEATURES] for (r, c) in coords_B]
    vec_mixed = np.mean(data[slice_idx, r_b:r_b + GRID_RATIO, c_b:c_b + GRID_RATIO, :N_FEATURES], axis=(0, 1))

    sig_A_raw = np.mean(vecs_A, axis=0) if len(vecs_A) else np.zeros(N_FEATURES, dtype=np.float64)
    sig_B_raw = np.mean(vecs_B, axis=0) if len(vecs_B) else np.zeros(N_FEATURES, dtype=np.float64)
    sig_M_raw = vec_mixed.astype(np.float64)

    # For visualization in panel (C)
    if PLOT_SPECTRA_RAW:
        sig_A, sig_B, sig_M = sig_A_raw, sig_B_raw, sig_M_raw
        y_label = "Intensity (raw units)"
    else:
        sig_A = robust_scale_01(sig_A_raw, SPECTRA_PCTL_LOW, SPECTRA_PCTL_HIGH)
        sig_B = robust_scale_01(sig_B_raw, SPECTRA_PCTL_LOW, SPECTRA_PCTL_HIGH)
        sig_M = robust_scale_01(sig_M_raw, SPECTRA_PCTL_LOW, SPECTRA_PCTL_HIGH)
        y_label = f"Intensity (robust scaled to [0,1], p{SPECTRA_PCTL_LOW}–p{SPECTRA_PCTL_HIGH})"

    # Pretty names (display only)
    raw_A = lut.get(int(target_labels[0]), f"Tissue A ({int(target_labels[0])})")
    raw_B = lut.get(int(target_labels[1]), f"Tissue B ({int(target_labels[1])})")

    name_A = pretty_label_name(raw_A, strip_hemi=STRIP_HEMISPHERE, strip_prefix=STRIP_PREFIX_WM_CTX)
    name_B = pretty_label_name(raw_B, strip_hemi=STRIP_HEMISPHERE, strip_prefix=STRIP_PREFIX_WM_CTX)

    # ===== Plot layout =====
    # Use constrained_layout to avoid tight_layout warnings
    fig = plt.figure(figsize=(18, 6), facecolor="white", constrained_layout=True)
    gs = gridspec.GridSpec(1, 3, width_ratios=[1, 1.2, 1.5], wspace=0.25)

    # (A) Full slice
    ax1 = fig.add_subplot(gs[0])
    ax1.imshow(img_disp, cmap="gray", origin="upper")
    ax1.add_patch(
        patches.Rectangle(
            (c_start, r_start),
            c_end - c_start,
            r_end - r_start,
            lw=1.5,
            edgecolor="lime",
            facecolor="none",
        )
    )
    ax1.add_patch(
        patches.Rectangle(
            (c_b, r_b),
            GRID_RATIO,
            GRID_RATIO,
            lw=2,
            edgecolor="red",
            facecolor="none",
        )
    )
    ax1.set_title("(A) Full Slice", fontweight="bold")
    ax1.axis("off")

    # (B) Cropped + label overlay + grid lines
    ax2 = fig.add_subplot(gs[1])
    ax2.imshow(img_crop, cmap="gray", origin="upper", extent=[c_start, c_end, r_end, r_start])

    overlay = np.zeros((lbl_crop.shape[0], lbl_crop.shape[1], 4), dtype=np.float32)
    overlay[lbl_crop == int(target_labels[0])] = [0, 0, 1, 0.40]  # blue
    overlay[lbl_crop == int(target_labels[1])] = [0, 1, 0, 0.40]  # green
    ax2.imshow(overlay, origin="upper", extent=[c_start, c_end, r_end, r_start])

    # Grid lines aligned to GRID_RATIO
    grid_sc = (c_start // GRID_RATIO) * GRID_RATIO
    grid_sr = (r_start // GRID_RATIO) * GRID_RATIO
    for c in range(grid_sc, c_end + 1, GRID_RATIO):
        ax2.axvline(c - 0.5, lw=0.5, alpha=0.5, color="cyan")
    for r in range(grid_sr, r_end + 1, GRID_RATIO):
        ax2.axhline(r - 0.5, lw=0.5, alpha=0.5, color="cyan")

    ax2.add_patch(
        patches.Rectangle(
            (c_b - 0.5, r_b - 0.5),
            GRID_RATIO,
            GRID_RATIO,
            lw=2.5,
            edgecolor="red",
            facecolor="none",
        )
    )
    ax2.set_title("(B) Mixed Block (Red Box)", fontweight="bold")
    ax2.axis("off")

    # Legend
    from matplotlib.lines import Line2D
    legs = [
        Line2D([0], [0], color="blue", lw=4, alpha=0.5),
        Line2D([0], [0], color="green", lw=4, alpha=0.5),
    ]
    ax2.legend(legs, [name_A, name_B], loc="lower center", bbox_to_anchor=(0.5, -0.15), fontsize=8)

    # (C) Spectral curves (no z-score)
    ax3 = fig.add_subplot(gs[2])
    x = np.arange(N_FEATURES, dtype=int)
    ax3.plot(x, sig_A, "-", alpha=0.6, lw=1, label=f"Pure '{name_A}'")
    ax3.plot(x, sig_B, "-", alpha=0.6, lw=1, label=f"Pure '{name_B}'")
    ax3.plot(x, sig_M, "-", alpha=1.0, lw=2, label="Mixed (block mean)")

    ax3.set_title("(C) Spectral Mixing (No z-score)", fontweight="bold")
    ax3.set_ylabel(y_label)
    ax3.set_xlabel("Feature Index")
    ax3.legend(loc="upper right", fontsize=8)

    # Modality boundary hints
    trans = ax3.get_xaxis_transform()
    ax3.text(7, -0.12, "QTI", transform=trans, ha="center", size=8)
    ax3.text(120, -0.12, "Diffusion", transform=trans, ha="center", size=8)
    ax3.text(280, -0.12, "CEST", transform=trans, ha="center", size=8)
    for l in [15, 225]:
        ax3.axvline(l, color="gray", ls="--", alpha=0.3)

    # y-limits
    if PLOT_SPECTRA_RAW:
        y_min = np.nanmin([sig_A.min(), sig_B.min(), sig_M.min()])
        y_max = np.nanmax([sig_A.max(), sig_B.max(), sig_M.max()])
        if np.isfinite(y_min) and np.isfinite(y_max) and (y_max - y_min) > 1e-12:
            pad = 0.05 * (y_max - y_min)
            ax3.set_ylim(y_min - pad, y_max + pad)
    else:
        ax3.set_ylim(-0.05, 1.05)

    plt.savefig(output_file, dpi=300)
    print(f"Generated: {output_file}")
    plt.close()


if __name__ == "__main__":
    lut = load_lut(LUT_CSV_PATH)
    data, labels = load_data_and_labels(DATA_PATH)
    plot_rev7(data, labels, lut, SLICE_IDX, OUTPUT_FILENAME)


Loading LUT: /home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx ...
Loaded 102 labels from LUT.
Loading Data: /home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat ...
Found 'region_labels' (Minimal Format).
  Data shape:   (384, 336, 256, 351)
  Labels shape: (384, 336, 256)


/tmp/ipykernel_28679/2923493500.py:433: UserWarning: There are no gridspecs with layoutgrids. Possibly did not call parent GridSpec with the "figure" keyword
  plt.savefig(output_file, dpi=300)


Generated: Figure_1_3_v7_FixedBlock_PrettyNames.png


In [12]:
fb, c_b = info["bbox"]
    ax1.add_patch(patches.Rectangle((cs, rs), ce-cs, re-rs, lw=1.5, edgecolor="lime", facecolor="none"))
    ax1.add_patch(patches.Rectangle((c_b, r_b), GRID_RATIO, GRID_RATIO, lw=2, edgecolor="red", facecolor="none"))
    ax1.set_title("(A) Full Slice", fontweight="bold")
    ax1.axis("off")


    # --- Panel B (Top Right) ---
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(info["img_crop"], cmap="gray", origin="upper", extent=info["crop_extent"])
    overlay = np.zeros((info["lbl_crop"].shape[0], info["lbl_crop"].shape[1], 4), dtype=np.float32)
    overlay[info["lbl_crop"] == info["labels"][0]] = [0, 0, 1, 0.40]
    overlay[info["lbl_crop"] == info["labels"][1]] = [0, 1, 0, 0.40]
    ax2.imshow(overlay, origin="upper", extent=info["crop_extent"])
    
    # Grid
    grid_sc = (cs // GRID_RATIO) * GRID_RATIO
    grid_sr = (rs // GRID_RATIO) * GRID_RATIO
    for c in range(grid_sc, ce + 1, GRID_RATIO): ax2.axvline(c - 0.5, lw=0.5, alpha=0.5, color="cyan")
    for r in range(grid_sr, re + 1, GRID_RATIO): ax2.axhline(r - 0.5, lw=0.5, alpha=0.5, color="cyan")
    ax2.add_patch(patches.Rectangle((c_b - 0.5, r_b - 0.5), GRID_RATIO, GRID_RATIO, lw=2.5, edgecolor="red", facecolor="none"))
    
    legs = [Line2D([0],[0], color="blue", lw=4, alpha=0.5), Line2D([0],[0], color="green", lw=4, alpha=0.5)]
    ax2.legend(legs, [info["name_A"], info["name_B"]], loc="lower right", fontsize=8)
    ax2.set_title("(B) Mixed Block Zoom", fontweight="bold")
    ax2.axis("off")


    # --- Panel C (Bottom Left) - Full Spectrum ---
    ax3 = fig.add_subplot(gs[1, 0])
    x = np.arange(N_FEATURES)
    ax3.plot(x, info["sig_A"], "-", alpha=0.6, lw=1, color="tab:blue", label=f"Pure A")
    ax3.plot(x, info["sig_B"], "-", alpha=0.6, lw=1, color="tab:green", label=f"Pure B")
    ax3.plot(x, info["sig_M"], "-", alpha=0.9, lw=1.2, color="tab:red", label="Mixed")
    
    # Highlight the zoom region
    ax3.axvspan(zoom_start, zoom_end, color='gold', alpha=0.2, label='Region in (D)')
    
    ax3.set_title("(C) Full Spectrum", fontweight="bold")
    ax3.set_xlabel("Feature Index")
    ax3.set_ylim(-0.05, 1.05) if not PLOT_SPECTRA_RAW else None
    ax3.legend(loc="upper right", fontsize=8)


    # --- Panel D (Bottom Right) - Zoomed Spectrum ---
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.plot(x, info["sig_A"], "-", alpha=0.6, lw=1.5, color="tab:blue", label=f"{info['name_A']}")
    ax4.plot(x, info["sig_B"], "-", alpha=0.6, lw=1.5, color="tab:green", label=f"{info['name_B']}")
    ax4.plot(x, info["sig_M"], "-", alpha=0.9, lw=2.0, color="tab:red", label="Mixed") # Slightly thicker here for clarity


    ax4.set_xlim(zoom_start, zoom_end)
    ax4.set_ylim(-0.05, 1.05) if not PLOT_SPECTRA_RAW else None
    
    # Highlight max difference point
    ax4.axvline(idx_max, color='black', linestyle='--', alpha=0.3)
    ax4.text(idx_max, 0.05, "Max Diff", rotation=90, verticalalignment='bottom', alpha=0.5, fontsize=8)
    
    # Background color to indicate zoom
    ax4.set_facecolor("#fffdf0") # Very light yellow


    ax4.set_title(f"(D) Zoom: Max Difference (Idx {idx_max})", fontweight="bold")
    ax4.set_xlabel("Feature Index")
    ax4.legend(loc="upper center", fontsize=8, ncol=1)


    plt.savefig(output_file, dpi=300)
    print(f"Generated 2x2 Layout: {output_file}")
    plt.close()


# ================= Main =================


if __name__ == "__main__":
    # 1. Load Data
    lut = load_lut(LUT_CSV_PATH)
    data, labels = load_data_and_labels(DATA_PATH)


    # 2. Determine Slice
    curr_slice = FIXED_SLICE_IDX if USE_FIXED_BLOCK else 160
    
    try:
        # 3. Process Data (Get signals, blocks, etc.)
        info_dict = get_block_data(data, labels, lut, curr_slice)
        
        # 4. Generate Plot 1: Original Layout (Thinner lines)
        plot_original_layout(info_dict, OUTPUT_FILE_ORIG)


        # 5. Generate Plot 2: 2x2 Layout (With Spectral Zoom)
        plot_2x2_layout(info_dict, OUTPUT_FILE_2x2)


    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

Loading LUT: /home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx ...
Loading Data: /home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat ...


/tmp/ipykernel_28679/675204284.py:286: UserWarning: There are no gridspecs with layoutgrids. Possibly did not call parent GridSpec with the "figure" keyword
  plt.savefig(output_file, dpi=300)


Generated Original Layout: Figure_1_3_v7_Original_ThinLine.png


/tmp/ipykernel_28679/675204284.py:371: UserWarning: There are no gridspecs with layoutgrids. Possibly did not call parent GridSpec with the "figure" keyword
  plt.savefig(output_file, dpi=300)


Generated 2x2 Layout: Figure_1_3_v7_2x2_Zoomed.png


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Figure 5.1 - Construction of Per-Voxel Soft Labels
-------------------------------------------------
Physics-based downsampling visualization (one-hot -> PSF smoothing -> linear downsample).
User supplies roi_anatomy (2D), roi_labels (2D int), lut_dict.
Parameters match batch_downsampling_pipeline v1.2.0:
- Input spacing (X,Y) = (218/336, 166/256) ≈ (0.649, 0.648) mm
- Target spacing (X,Y) = (1.8, 1.8) mm
- PSF sigma_add_mm (MPRAGE family, linear) = (0.713, 0.713) mm (z not used here)
- Classes fixed to 102 (0-101) unless explicitly overridden
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
from scipy.ndimage import gaussian_filter, zoom

# =============================================================================
# Physical parameters (consistent with batch_downsampling_pipeline v1.2.0)
# =============================================================================
SPACING_IN_XY = (218/336, 166/256)  # (X, Y) mm from ChannelConfig.INPUT_SPACING_MM
SPACING_OUT_XY = (1.8, 1.8)        # Target CEST in-plane spacing
SIGMA_ADD_MM_XY = (0.713, 0.713)    # MPRAGE sigma_add_mm (linear for probability labels)
NUM_CLASSES_DEFAULT = 102

SIGMA_ADD_PX = (
    SIGMA_ADD_MM_XY[0] / SPACING_IN_XY[0],
    SIGMA_ADD_MM_XY[1] / SPACING_IN_XY[1]
)

ZOOM_FACTORS = (
    SPACING_IN_XY[0] / SPACING_OUT_XY[0],  # y-direction uses spacing_x (pipeline X)
    SPACING_IN_XY[1] / SPACING_OUT_XY[1],  # x-direction uses spacing_y (pipeline Y)
    1.0                                    # channels
)

# =============================================================================
# LUT helper (consistent with fig1-3.ipynb style)
# =============================================================================
import pandas as pd

def load_lut(file_path):
    """Load label->name/color mapping from .xlsx/.csv with flexible columns."""
    print(f"Loading LUT: {file_path} ...")
    try:
        if file_path.endswith((".xlsx", ".xls")):
            df = pd.read_excel(file_path)
        else:
            df = pd.read_csv(file_path, sep=None, engine="python")

        def find_col(candidates):
            for c in candidates:
                if c in df.columns:
                    return c
            return None

        col_id = find_col(["one_hot_loc_alex_label", "freesurfer_label", "idx"])
        col_name = find_col(["freesurfer_tissue_name", "tissue_name", "col label"])
        col_r = find_col(["R", "r"])
        col_g = find_col(["G", "g"])
        col_b = find_col(["B", "b"])

        if (col_id is None) or (col_name is None):
            print(f"[Warn] LUT columns not found. Available: {list(df.columns)}")
            return {}

        lut = {}
        valid = pd.to_numeric(df[col_id], errors="coerce").notna()
        ids = df.loc[valid, col_id].astype(int)
        names = df.loc[valid, col_name].astype(str)
        colors = None
        if col_r and col_g and col_b:
            colors = df.loc[valid, [col_r, col_g, col_b]].astype(float).values

        for idx, name in zip(ids, names):
            entry = {"name": name}
            if colors is not None:
                vec = colors[len(lut)]
                entry.update({"R": vec[0], "G": vec[1], "B": vec[2]})
            lut[int(idx)] = entry

        print(f"Loaded {len(lut)} labels from LUT.")
        return lut
    except Exception as e:
        print(f"[Warn] Failed to load LUT: {e}")
        return {}

# =============================================================================
# Core processing: one-hot -> PSF smoothing -> linear downsample
# =============================================================================

def _normalize_prob(p):
    s = p.sum(axis=-1, keepdims=True)
    s = np.where(s > 0, s, 1.0)
    return p / s

def build_color_table(lut_dict, num_classes):
    cmap_fallback = plt.cm.tab20
    colors = np.zeros((num_classes, 3), dtype=float)
    for i in range(num_classes):
        entry = lut_dict.get(i)
        if isinstance(entry, dict):
            r = entry.get("R") or entry.get("r")
            g = entry.get("G") or entry.get("g")
            b = entry.get("B") or entry.get("b")
            if r is not None and g is not None and b is not None:
                vec = np.array([r, g, b], dtype=float)
                colors[i] = vec / (255.0 if vec.max() > 1.0 else 1.0)
                continue
            if "color" in entry:
                vec = np.array(entry["color"], dtype=float)
                colors[i] = vec[:3] / (255.0 if vec.max() > 1.0 else 1.0)
                continue
        elif isinstance(entry, (list, tuple)) and len(entry) >= 3:
            vec = np.array(entry[:3], dtype=float)
            colors[i] = vec / (255.0 if vec.max() > 1.0 else 1.0)
            continue
        colors[i] = np.array(cmap_fallback(i % cmap_fallback.N)[:3])
    return colors

def lut_name(label, lut_dict):
    entry = lut_dict.get(int(label))
    if isinstance(entry, dict):
        for k in ("freesurfer_tissue_name", "tissue_name", "name", "label"):
            if k in entry:
                return str(entry[k])
    return str(entry) if entry is not None else f"Label {label}"

def one_hot_encode(labels_2d, num_classes):
    h, w = labels_2d.shape
    one_hot = np.zeros((h, w, num_classes), dtype=np.float32)
    for c in range(num_classes):
        mask = (labels_2d == c)
        if mask.any():
            one_hot[..., c] = mask.astype(np.float32)
    return one_hot

def process_roi(roi_labels, num_classes=None):
    """One-hot -> anisotropic Gaussian (xy) -> linear downsample to target grid."""
    max_label = int(np.max(roi_labels))
    num_classes = num_classes or max(NUM_CLASSES_DEFAULT, max_label + 1)

    one_hot = one_hot_encode(roi_labels, num_classes)

    # Step 2: PSF smoothing in physical space (xy only for this slice)
    soft_hr = gaussian_filter(one_hot, sigma=(SIGMA_ADD_PX[0], SIGMA_ADD_PX[1], 0.0), mode="constant")
    soft_hr = _normalize_prob(soft_hr)

    # Step 3: Downsample to target spacing (linear)
    soft_lr = zoom(soft_hr, zoom=ZOOM_FACTORS, order=1, mode="reflect")
    soft_lr = _normalize_prob(soft_lr)

    return soft_hr.astype(np.float32), soft_lr.astype(np.float32), num_classes

# =============================================================================
# Visualization
# =============================================================================

def pick_mixed_voxel(proba_lr):
    entropy = -np.sum(proba_lr * np.log(proba_lr + 1e-8), axis=-1)
    y, x = np.unravel_index(np.argmax(entropy), entropy.shape)
    return int(y), int(x), proba_lr[y, x]

def prob_to_rgb(prob_map, colors):
    return np.tensordot(prob_map, colors, axes=([2], [0]))

def plot_figure_5_1(roi_anatomy, roi_labels, lut_dict):
    assert roi_anatomy.shape == roi_labels.shape, "roi_anatomy and roi_labels must align"

    soft_hr, soft_lr, num_classes = process_roi(roi_labels)
    colors = build_color_table(lut_dict, num_classes)

    # Physical extents for grid overlay (mm)
    h_hr, w_hr = roi_labels.shape
    width_mm = w_hr * SPACING_IN_XY[1]  # width aligns with Y spacing (pipeline Y)
    height_mm = h_hr * SPACING_IN_XY[0] # height aligns with X spacing (pipeline X)

    rgb_hr = prob_to_rgb(soft_hr, colors)
    rgb_lr = prob_to_rgb(soft_lr, colors)

    vy, vx, voxel_prob = pick_mixed_voxel(soft_lr)
    h_lr, w_lr, _ = soft_lr.shape
    voxel_w_mm = width_mm / w_lr
    voxel_h_mm = height_mm / h_lr

    fig = plt.figure(figsize=(18, 4.8), facecolor="white")
    gs = gridspec.GridSpec(1, 4, width_ratios=[1, 1, 1, 1.05], wspace=0.18)

    # Panel A: High-res hard labels
    axA = fig.add_subplot(gs[0])
    axA.imshow(roi_anatomy, cmap="gray", origin="upper")
    axA.imshow(roi_labels, cmap=ListedColormap(colors), origin="upper", alpha=0.55,
               interpolation="nearest", vmin=0, vmax=num_classes-1)
    axA.set_title("A. High-Res Hard Labels\n(0.65mm Isotropic)", fontweight="bold")
    axA.axis("off")

    # Panel B: PSF smoothing (probability blend)
    axB = fig.add_subplot(gs[1])
    axB.imshow(roi_anatomy, cmap="gray", origin="upper", alpha=0.25)
    axB.imshow(rgb_hr, origin="upper")
    axB.set_title("B. Physical Smoothing\n(Simulating PSF)", fontweight="bold")
    axB.axis("off")

    # Panel C: Native CEST grid (pixelated) + grid lines + cyan box
    axC = fig.add_subplot(gs[2])
    extent = [0, width_mm, height_mm, 0]
    axC.imshow(rgb_lr, origin="upper", interpolation="nearest", extent=extent)

    for gx in np.arange(0, width_mm + 1e-6, SPACING_OUT_XY[1]):
        axC.axvline(gx, color="white", lw=0.6, alpha=0.9)
    for gy in np.arange(0, height_mm + 1e-6, SPACING_OUT_XY[0]):
        axC.axhline(gy, color="white", lw=0.6, alpha=0.9)

    rect = plt.Rectangle((vx * voxel_w_mm, vy * voxel_h_mm), voxel_w_mm, voxel_h_mm,
                         edgecolor="cyan", facecolor="none", lw=2.0)
    axC.add_patch(rect)
    axC.set_title("C. Native CEST Grid\n(1.8mm × 1.8mm)", fontweight="bold")
    axC.set_xticks([])
    axC.set_yticks([])

    # Panel D: Voxel signature (top-4 probabilities)
    axD = fig.add_subplot(gs[3])
    top_idx = np.argsort(voxel_prob)[::-1][:4]
    top_vals = voxel_prob[top_idx]
    bar_colors = colors[top_idx]
    names = [lut_name(int(i), lut_dict) for i in top_idx]

    axD.bar(range(len(top_idx)), top_vals, color=bar_colors, edgecolor="k")
    axD.set_xticks(range(len(top_idx)))
    axD.set_xticklabels(names, rotation=20, ha="right")
    axD.set_ylim(0, 1.0)
    axD.set_ylabel("Probability")
    axD.set_title("D. Soft Voxel Signature\n(Target voxel)", fontweight="bold")

    plt.tight_layout()
    return fig



# ==== Core processing (one-hot -> PSF -> linear downsample) ===


In [ ]:
if __name__ == "__main__":
    # 用户负责加载 roi_anatomy (2D灰度), roi_labels (2D int), lut_dict。
    # 示例：
    # roi_anatomy = ... # shape (H, W)
    # roi_labels = ...  # shape (H, W)
    # lut_dict = load_lut("/path/to/Freesurfer_LUT_alex_labels_jiayi.xlsx")
    # fig = plot_figure_5_1(roi_anatomy, roi_labels, lut_dict)
    # fig.savefig("Figure_5_1_per_voxel_soft_labels.png", dpi=300)
    # plt.close(fig)
    print("Define roi_anatomy, roi_labels, lut_dict, then call plot_figure_5_1(...)")


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Figure 5.1 Execution Cell
--------------------------
完整流程：加载MAT + LUT -> 选择切片 -> 物理PSF概率下采样 -> 绘制4列布局并保存。
运行前请检查路径配置。
"""

import h5py
from pathlib import Path

# ================= 路径配置 =================
# 修改为你的真实路径
DATA_PATH = Path("/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/3D_minimal/ODP_01_qhlazec_3d_validated_minimal.mat")
LUT_PATH = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx")
OUTPUT_FIG = Path("Figure_5_1_per_voxel_soft_labels.png")
SLICE_IDX = 222
ANATOMY_CHANNEL = 341  # MPRAGE channel index
CROP_MARGIN = 20       # 像素：对非零标签 bbox 扩展以缩小显示区域

# ================= 数据加载 =================
def load_data_and_labels(mat_path: Path):
    """加载4D数据+标签，维度转为 (X, Y, Z, C)。"""
    print(f"Loading data: {mat_path} ...")
    with h5py.File(mat_path, "r") as f:
        if "data" not in f:
            raise ValueError("MAT 文件缺少 data")
        data = f["data"][:]
        if data.shape[0] in [341, 351]:
            data = np.moveaxis(data, 0, -1)
        if "region_labels" in f:
            labels = f["region_labels"][:]
        elif "one_hot_loc_alex_label" in f:
            oh = f["one_hot_loc_alex_label"][:]
            axis = 0 if oh.shape[0] == 102 else -1
            labels = np.argmax(oh, axis=axis)
        else:
            raise ValueError("MAT 文件缺少 region_labels/one_hot_loc_alex_label")
    print(f"  data shape: {data.shape}, labels shape: {labels.shape}")
    return data.astype(np.float32), labels.astype(int)

def crop_to_nonzero(lbl2d, margin=0):
    ys, xs = np.nonzero(lbl2d)
    if len(ys) == 0:
        return (0, lbl2d.shape[0], 0, lbl2d.shape[1])
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    y0 = max(0, y0 - margin); y1 = min(lbl2d.shape[0], y1 + margin)
    x0 = max(0, x0 - margin); x1 = min(lbl2d.shape[1], x1 + margin)
    return (y0, y1, x0, x1)

# ================= 主流程 =================
def run_figure_5_1():
    # 1) 读 LUT
    lut_dict = load_lut(str(LUT_PATH))

    # 2) 读数据/标签，取切片
    data, labels = load_data_and_labels(DATA_PATH)
    assert 0 <= SLICE_IDX < data.shape[0], "SLICE_IDX 越界"
    img_slice = data[SLICE_IDX, :, :, ANATOMY_CHANNEL]
    lbl_slice = labels[SLICE_IDX, :, :]

    # 3) 裁剪到非零标签区域（可选）
    y0, y1, x0, x1 = crop_to_nonzero(lbl_slice, margin=CROP_MARGIN)
    img_roi = img_slice[y0:y1, x0:x1]
    lbl_roi = lbl_slice[y0:y1, x0:x1]

    # 4) 生成图
    fig = plot_figure_5_1(img_roi, lbl_roi, lut_dict)
    fig.savefig(OUTPUT_FIG, dpi=300)
    plt.close(fig)
    print(f"✅ Figure saved: {OUTPUT_FIG}")

run_figure_5_1()
